# RFP Generation Module Colab Experiment

이 노트북은 새로 받은 `src/generation/rfp_generation.py` 모듈을 기준으로 generation을 다시 실행합니다.

핵심 방향:
- 690 retrieval JSONL은 고정합니다.
- Chroma/vector DB는 사용하지 않습니다.
- `rfp_generation.py`의 field-aware context package, intent plan, deterministic calculation, postprocess, review output 저장 로직을 사용합니다.
- 기본값은 `source_store=False`입니다.
- 사람이 볼 결과는 `llm_answer_review.html`, `review_samples.csv`, `metrics_summary.json`입니다.


In [ ]:
# 1. Experiment config
from pathlib import Path

REPO_URL = 'https://github.com/beomsookim1020/chatbot.git'
REPO_BRANCH = 'colab-generation'
PROJECT_DIR = Path('/content/chatbot')

DRIVE_INPUT_ROOT = Path('/content/drive/MyDrive/chatbot_colab_inputs')
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/chatbot_colab_outputs')
DRIVE_EXPERIMENT_ROOT = DRIVE_OUTPUT_ROOT / 'generation_rfp_module_experiments'

EXPERIMENT_ID = 'rfp_module_690_clean'
EXPERIMENT_NAME = 'rfp_module_690_field_aware'

PREDICTION_REL = Path(
    'outputs/predictions/91_dense_qdecomp_rrf_per75_docscore_mean3_300_kure_chroma_690_canonical.jsonl'
)
CHUNK_SIDECAR_REL = Path('indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl')
EVAL_REL = Path('data/eval/representative_wrong_30_eval_batch_format.csv')
SOURCE_STORE_REL = Path('data/source_store_v2_690.jsonl')

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
RUN_LIMIT = 0
EXPERIMENT_IDS = []

MAX_NEW_TOKENS = 1024
TEMPERATURE = 0.0
TOP_P = 1.0

USE_SOURCE_STORE = False
REVIEW_FOCUS = False
RANDOM_SEED = 42

GENERATION_CONFIG = {
    'use_source_store': USE_SOURCE_STORE,
    'max_context_chars_fact': 9000,
    'max_context_chars_synthesis': 12000,
    'max_blocks_fact': 8,
    'max_blocks_synthesis': 12,
    'evidence_text_chars': 1100,
    'source_store_text_chars': 1400,
}

print('experiment:', EXPERIMENT_NAME)
print('prediction:', PREDICTION_REL)
print('chunks:', CHUNK_SIDECAR_REL)
print('eval:', EVAL_REL)
print('model:', MODEL_NAME)


experiment: rfp_module_690_field_aware
prediction: outputs/predictions/91_dense_qdecomp_rrf_per75_docscore_mean3_300_kure_chroma_690_canonical.jsonl
chunks: indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl
eval: data/eval/representative_wrong_30_eval_batch_format.csv
model: Qwen/Qwen2.5-3B-Instruct


In [2]:
# 2. Runtime setup: GPU, Drive, repo, packages
import shutil
import subprocess
import sys

if shutil.which('nvidia-smi') is None:
    raise RuntimeError('Colab GPU runtime이 아닙니다. Runtime > Change runtime type > GPU로 바꿔주세요.')
subprocess.run(['nvidia-smi'], check=True)

from google.colab import drive
drive.mount('/content/drive')

def run_cmd(cmd, cwd=None):
    print('$', ' '.join(map(str, cmd)))
    subprocess.run([str(part) for part in cmd], cwd=str(cwd) if cwd else None, check=True)

if (PROJECT_DIR / '.git').exists():
    run_cmd(['git', 'fetch', 'origin', REPO_BRANCH], cwd=PROJECT_DIR)
    run_cmd(['git', 'checkout', REPO_BRANCH], cwd=PROJECT_DIR)
    run_cmd(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=PROJECT_DIR)
else:
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    run_cmd(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, PROJECT_DIR])

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.45.0', 'accelerate', 'sentencepiece', 'safetensors'
], check=True)

ROOT = str(PROJECT_DIR)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

MODULE_PATH = PROJECT_DIR / 'src/generation/rfp_generation.py'
if not MODULE_PATH.exists():
    raise FileNotFoundError(
        f'{MODULE_PATH}가 없습니다. rfp_generation.py를 repo에 추가한 뒤 GitHub에 push하고 다시 실행해주세요.'
    )

print('Runtime ready.')


Mounted at /content/drive
$ git clone --branch colab-generation --single-branch https://github.com/beomsookim1020/chatbot.git /content/chatbot


FileNotFoundError: /content/chatbot/src/generation/rfp_generation.py가 없습니다. rfp_generation.py를 repo에 추가한 뒤 GitHub에 push하고 다시 실행해주세요.

In [ ]:
# 3. Copy inputs from Drive
import shutil

def first_existing(paths):
    return next((path for path in paths if path and path.exists()), None)

DRIVE_EVAL = DRIVE_INPUT_ROOT / EVAL_REL
DRIVE_PREDICTIONS = DRIVE_INPUT_ROOT / PREDICTION_REL
DRIVE_CHUNKS = first_existing([
    DRIVE_INPUT_ROOT / CHUNK_SIDECAR_REL,
    (DRIVE_INPUT_ROOT / CHUNK_SIDECAR_REL).with_suffix('.json'),
])
DRIVE_SOURCE_STORE = DRIVE_INPUT_ROOT / SOURCE_STORE_REL

missing = [str(path) for path in [DRIVE_EVAL, DRIVE_PREDICTIONS] if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required inputs:\n' + '\n'.join(missing))

if DRIVE_CHUNKS is None:
    print('WARN: chunk sidecar를 찾지 못했습니다. retrieved_contexts만 사용합니다.')
if USE_SOURCE_STORE and not DRIVE_SOURCE_STORE.exists():
    print('WARN: USE_SOURCE_STORE=True지만 source_store 파일이 없습니다. source_store는 비활성화됩니다.')

LOCAL_EVAL = PROJECT_DIR / EVAL_REL
LOCAL_PREDICTIONS = PROJECT_DIR / PREDICTION_REL
LOCAL_CHUNKS = PROJECT_DIR / CHUNK_SIDECAR_REL if DRIVE_CHUNKS else None
LOCAL_SOURCE_STORE = PROJECT_DIR / SOURCE_STORE_REL if DRIVE_SOURCE_STORE.exists() else None

LOCAL_EVAL.parent.mkdir(parents=True, exist_ok=True)
LOCAL_PREDICTIONS.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(DRIVE_EVAL, LOCAL_EVAL)
shutil.copy2(DRIVE_PREDICTIONS, LOCAL_PREDICTIONS)

if DRIVE_CHUNKS and LOCAL_CHUNKS:
    LOCAL_CHUNKS.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_CHUNKS, LOCAL_CHUNKS)
if DRIVE_SOURCE_STORE.exists() and LOCAL_SOURCE_STORE:
    LOCAL_SOURCE_STORE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_SOURCE_STORE, LOCAL_SOURCE_STORE)

print('eval:', LOCAL_EVAL)
print('predictions:', LOCAL_PREDICTIONS)
print('chunks:', LOCAL_CHUNKS)
print('source_store:', LOCAL_SOURCE_STORE)


In [ ]:
# 4. Import rfp_generation module and prepare items
import csv
import json
import time
from datetime import datetime, timezone

from src.generator import HuggingFaceGenerator
from src.generation.rfp_generation import (
    build_context_package,
    build_prompt,
    enrich_generation_record,
    load_chunk_index,
    load_generation_input_rows,
    load_source_store_index,
    postprocess_answer,
    prepare_generation_items,
    read_csv_records,
    save_generation_outputs,
)

result_rows, context_rows = load_generation_input_rows(
    LOCAL_PREDICTIONS,
    LOCAL_PREDICTIONS,
    experiment_id=EXPERIMENT_ID,
)

eval_rows = read_csv_records(LOCAL_EVAL)
eval_by_id = {str(row.get('id')): row for row in eval_rows}
for row in result_rows:
    qid = str(row.get('id') or row.get('question_id'))
    eval_row = eval_by_id.get(qid, {})
    if eval_row:
        row['question'] = eval_row.get('question') or row.get('question')
        row['ground_truth_answer'] = eval_row.get('ground_truth_answer', '')
        row['ground_truth_docs'] = eval_row.get('ground_truth_docs', '')
        row['type'] = eval_row.get('type', '')
        row['difficulty'] = eval_row.get('difficulty', '')
        row['metadata_filter'] = eval_row.get('metadata_filter', '')

sample_size = None if RUN_LIMIT == 0 else RUN_LIMIT
items = prepare_generation_items(
    result_rows,
    context_rows,
    experiment_id=EXPERIMENT_ID,
    sample_size=sample_size,
    review_focus=REVIEW_FOCUS,
    random_seed=RANDOM_SEED,
)
if EXPERIMENT_IDS:
    keep = set(EXPERIMENT_IDS)
    items = [item for item in items if item.get('question_id') in keep]

chunk_ids = {
    str(context.get('chunk_id'))
    for item in items
    for context in item.get('retrieved_contexts', [])
    if context.get('chunk_id')
}
source_files = {
    str(context.get('source_file') or context.get('filename'))
    for item in items
    for context in item.get('retrieved_contexts', [])
    if context.get('source_file') or context.get('filename')
}

chunk_index = {}
if LOCAL_CHUNKS and LOCAL_CHUNKS.exists():
    try:
        chunk_index = load_chunk_index(
            LOCAL_CHUNKS,
            chunk_ids=chunk_ids or None,
            source_files=source_files or None,
        )
    except Exception as exc:
        print('WARN: chunk index 로딩에 실패했습니다. retrieved_contexts만 사용합니다:', repr(exc))
        chunk_index = {}

source_store_index = load_source_store_index(
    LOCAL_SOURCE_STORE,
    enabled=bool(USE_SOURCE_STORE and LOCAL_SOURCE_STORE and LOCAL_SOURCE_STORE.exists()),
)

print('result rows:', len(result_rows))
print('context rows:', len(context_rows))
print('items:', len(items))
print('chunk index:', len(chunk_index))
print('source_store index:', len(source_store_index))
print('first ids:', [item.get('question_id') for item in items[:5]])


In [ ]:
# 5. Dry-run context package preview
if not items:
    raise RuntimeError('No generation items selected.')

preview_item = items[0]
preview_package = build_context_package(
    preview_item['question'],
    preview_item['retrieved_contexts'],
    chunk_index=chunk_index,
    source_store_index=source_store_index,
    use_source_store=bool(USE_SOURCE_STORE and source_store_index),
    config=GENERATION_CONFIG,
)
preview_messages = build_prompt(preview_package)

print('question_id:', preview_item['question_id'])
print('question:', preview_item['question'])
print('analysis:', json.dumps(preview_package.get('question_analysis', {}), ensure_ascii=False, indent=2)[:3000])
print('evidence_blocks:', len(preview_package.get('evidence_blocks', [])))
print('context preview:')
print(preview_package.get('context_text', '')[:4000])
print('prompt roles:', [message['role'] for message in preview_messages])


In [ ]:
# 6. Run generation
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
LOCAL_OUTPUT_DIR = PROJECT_DIR / 'outputs/generation_rfp_module' / RUN_TIMESTAMP
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

generator = HuggingFaceGenerator(
    model_name=MODEL_NAME,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
)

def run_messages(messages):
    system_prompt = ''
    user_prompt = ''
    for message in messages:
        if message.get('role') == 'system':
            system_prompt = message.get('content', '')
        elif message.get('role') == 'user':
            user_prompt = message.get('content', '')
    return generator.generate_prompt(user_prompt, system_prompt=system_prompt or None)

records = []
for index, item in enumerate(items, 1):
    started = time.perf_counter()
    context_package = build_context_package(
        item['question'],
        item['retrieved_contexts'],
        chunk_index=chunk_index,
        source_store_index=source_store_index,
        use_source_store=bool(USE_SOURCE_STORE and source_store_index),
        config=GENERATION_CONFIG,
    )
    messages = build_prompt(context_package)
    raw_text = run_messages(messages)
    answer = postprocess_answer(raw_text, context_package)
    generation_ms = int((time.perf_counter() - started) * 1000)
    record = enrich_generation_record(
        answer,
        item,
        context_package,
        generation_ms=generation_ms,
        model_name=MODEL_NAME,
        experiment_name=EXPERIMENT_NAME,
        run_timestamp=RUN_TIMESTAMP,
    )
    record['_raw_text'] = raw_text
    records.append(record)
    print(
        f"{index}/{len(items)} {item['question_id']} type={record.get('answer_type')} "
        f"status={record.get('answer_status')} tags={record.get('_failure_tags')}"
    )

print('Local output dir:', LOCAL_OUTPUT_DIR)


In [ ]:
# 7. Save outputs and copy to Drive
run_config = {
    'experiment_name': EXPERIMENT_NAME,
    'experiment_id': EXPERIMENT_ID,
    'model_name': MODEL_NAME,
    'prediction': str(PREDICTION_REL),
    'chunks': str(CHUNK_SIDECAR_REL),
    'eval': str(EVAL_REL),
    'run_limit': RUN_LIMIT,
    'use_source_store': USE_SOURCE_STORE,
    'generation_config': GENERATION_CONFIG,
    'run_timestamp': RUN_TIMESTAMP,
}

paths = save_generation_outputs(
    LOCAL_OUTPUT_DIR,
    records,
    run_config=run_config,
)

DRIVE_OUTPUT_DIR = DRIVE_EXPERIMENT_ROOT / RUN_TIMESTAMP
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for path in LOCAL_OUTPUT_DIR.glob('*'):
    if path.suffix.lower() in {'.jsonl', '.csv', '.html', '.json', '.md'}:
        shutil.copy2(path, DRIVE_OUTPUT_DIR / path.name)

print('Local output dir:', LOCAL_OUTPUT_DIR)
print('Drive output dir:', DRIVE_OUTPUT_DIR)
print('saved paths:')
for key, value in paths.items():
    print('-', key, value)


## 결과 보는 순서

Drive 결과 경로:

`MyDrive/chatbot_colab_outputs/generation_rfp_module_experiments/{RUN_TIMESTAMP}/`

먼저 볼 파일:

1. `metrics_summary.json`: valid JSON, empty answer, numeric grounding, failure tag 요약
2. `failure_tags_summary.json`: 실패 태그별 예시
3. `llm_answer_review.html`: raw LLM text / parsed answer / final answer / GT 비교
4. `review_samples.csv`: 사람이 직접 `failure_type`, `review_memo`를 채울 검토용 CSV
5. `generated_answers.jsonl`: 전체 record와 context diagnostics
